# AMJax — Benchmark

Runs all benchmark configurations defined in the YAML config file.
For each combination of parameters the cartesian product is computed, one benchmark run is executed per combination, and results are saved as a JSON file in `results/`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vboussange/AMJax/blob/main/benchmarks/benchmark.ipynb)

In [2]:
CONFIG_FILE = "config_local.yaml"

In [ ]:
import timeit
from itertools import product as cartesian_product

import numpy as np
import yaml
import jax
import jax.numpy as jnp
import pyamg
from pyamg.relaxation.smoothing import change_smoothers
from scipy.sparse.linalg import cg

from amjax import MultilevelSolver
from benchmarks.plots import save_results

import sys
sys.path.insert(0, "..")

jax.config.update("jax_enable_x64", True)
np.random.seed(42)

In [4]:
with open(CONFIG_FILE) as f:
    config = yaml.safe_load(f)

factor_keys = ["solvers", "coarse_solvers", "tol", "maxiter_vcycle", "maxiter_solv", "vmap_k", "dtypes", "cycle_type", "grid_sizes", "methods", "smoothers"]
configurations = list(cartesian_product(*[config[k] for k in factor_keys]))

print(f"JAX backend: {jax.devices()[0].platform}")
print(f"Configurations: {len(configurations)}")


JAX backend: cpu
Configurations: 4


In [5]:
SOLVERS = {
    "smoothed_aggregation": pyamg.smoothed_aggregation_solver,
    "rootnode": pyamg.rootnode_solver,
    "pairwise": pyamg.pairwise_solver,
    "ruge_stuben": pyamg.ruge_stuben_solver,
    "air": pyamg.air_solver,
}

## Solver functions

In [6]:
def make_solve_fns(tol, maxiter_vcycle, maxiter_solv, cycle_type):
    @jax.jit
    def amjax_solve(ml, b):
        x = ml.solve(b, tol=tol, maxiter=maxiter_vcycle, cycle=cycle_type)
        error = jnp.linalg.norm(b - ml.levels[0].A @ x) / jnp.linalg.norm(b)
        return x, error

    @jax.jit
    def amjax_pcg_solve(ml, b):
        M = ml.aspreconditioner()
        x, _ = jax.scipy.sparse.linalg.cg(ml.levels[0].A, b, M=M, tol=tol, maxiter=maxiter_solv)
        error = jnp.linalg.norm(b - ml.levels[0].A @ x) / jnp.linalg.norm(b)
        return x, error

    def pyamg_solve(ml, b):
        x = ml.solve(b, tol=tol, maxiter=maxiter_vcycle, cycle=cycle_type)
        error = np.linalg.norm(b - ml.levels[0].A @ x) / np.linalg.norm(b)
        return x, error

    def pyamg_pcg_solve(ml, b):
        M = ml.aspreconditioner()
        x, _ = cg(ml.levels[0].A, b, M=M, rtol=tol, maxiter=maxiter_solv)
        error = np.linalg.norm(b - ml.levels[0].A @ x) / np.linalg.norm(b)
        return x, error

    def scipy_cg_solve(A, b):
        x, _ = cg(A, b, rtol=tol, maxiter=maxiter_solv)
        error = np.linalg.norm(b - A @ x) / np.linalg.norm(b)
        return x, error

    return amjax_solve, amjax_pcg_solve, pyamg_solve, pyamg_pcg_solve, scipy_cg_solve

In [7]:
def make_batch_solvers(amjax_solve, amjax_pcg_solve, pyamg_solve, pyamg_pcg_solve, scipy_cg_solve):

    def pyamg_batch(ml, B):
        results = [pyamg_solve(ml, B[i]) for i in range(len(B))]
        return np.array([x for x, _ in results]), float(np.mean([e for _, e in results]))

    def pyamg_pcg_batch(ml, B):
        results = [pyamg_pcg_solve(ml, B[i]) for i in range(len(B))]
        return np.array([x for x, _ in results]), float(np.mean([e for _, e in results]))

    def cg_batch(A, B):
        results = [scipy_cg_solve(A, B[i]) for i in range(len(B))]
        return np.array([x for x, _ in results]), float(np.mean([e for _, e in results]))

    def amjax_batch(ml, B):
        results = [amjax_solve(ml, B[i]) for i in range(len(B))]
        return np.array([x for x, _ in results]), float(np.mean([e for _, e in results]))

    def amjax_pcg_batch(ml, B):
        results = [amjax_pcg_solve(ml, B[i]) for i in range(len(B))]
        return np.array([x for x, _ in results]), float(np.mean([e for _, e in results]))

    def amjax_vmap_batch(ml, B):
        results = jax.vmap(lambda b: amjax_solve(ml, b))(B)
        return results[0], float(jnp.mean(results[1]))

    def amjax_pcg_vmap_batch(ml, B):
        results = jax.vmap(lambda b: amjax_pcg_solve(ml, b))(B)
        return results[0], float(jnp.mean(results[1]))

    return {
        "pyamg": pyamg_batch,
        "pyamg_pcg": pyamg_pcg_batch,
        "cg": cg_batch,
        "amjax": amjax_batch,
        "amjax_pcg": amjax_pcg_batch,
        "amjax_vmap": amjax_vmap_batch,
        "amjax_pcg_vmap": amjax_pcg_vmap_batch,
    }


def benchmark(method, func, is_jax=False):
    if is_jax:
        jax.block_until_ready(func())
    times = timeit.repeat(func, number=1, repeat=10)
    _, error = func()
    print(f"{method}/ time: {min(times):.4f}s/ residual: {float(error):.2e}")
    return min(times), float(error)


def cast_solver(ml, dtype):
    for lvl in ml.levels:
        lvl.A = lvl.A.astype(dtype)
        lvl.Dinv = lvl.Dinv.astype(dtype)
        if lvl.P is not None: lvl.P = lvl.P.astype(dtype)
        if lvl.R is not None: lvl.R = lvl.R.astype(dtype)
    return ml


## Benchmark

In [8]:
JAX_METHODS = {"amjax", "amjax_pcg", "amjax_vmap", "amjax_pcg_vmap"}

for combo in configurations:
    solver_name, coarse_solver, tol, maxiter_vcycle, maxiter_solv, vmap_k, dtype, cycle_type, grid_size, method, smoother = combo
    jnp_dtype = jnp.float32 if dtype == "f32" else jnp.float64

    print(f"\nConfig: {solver_name} | {coarse_solver} | {dtype} | cycle={cycle_type} | smoother={smoother} | n={grid_size} | {method}")

    A = pyamg.gallery.poisson((grid_size, grid_size), format="csr")
    B = np.random.rand(vmap_k, A.shape[0])
    B_jax = jnp.array(B, dtype=jnp_dtype)

    try:
        ml_pyamg = SOLVERS[solver_name](A, coarse_solver=coarse_solver)
        change_smoothers(ml_pyamg, presmoother=smoother, postsmoother=smoother)
        ml_jax = MultilevelSolver.from_pyamg(
            ml_pyamg,
            presmoother=(smoother, {"iterations": 1, "withrho": True}),
            postsmoother=(smoother, {"iterations": 1, "withrho": True}),
        )
        if dtype == "f32":
            ml_jax = cast_solver(ml_jax, jnp.float32)
    except Exception as exc:
        print(f"  ERROR building solver: {exc}")
        continue

    inputs = {
        "pyamg": (ml_pyamg, B),
        "pyamg_pcg": (ml_pyamg, B),
        "cg": (A, B),
        "amjax": (ml_jax, B_jax),
        "amjax_pcg": (ml_jax, B_jax),
        "amjax_vmap": (ml_jax, B_jax),
        "amjax_pcg_vmap":(ml_jax, B_jax),
    }

    batch_solver = make_batch_solvers(*make_solve_fns(tol, maxiter_vcycle, maxiter_solv, cycle_type))[method]

    try:
        solve = lambda: batch_solver(*inputs[method])
        time, residual = benchmark(f"{method}-{solver_name}", solve, is_jax=method in JAX_METHODS)
    except Exception as exc:
        print(f"  ERROR: {exc}")
        time, residual = float("nan"), float("nan")

    config_name = f"{solver_name}_{coarse_solver}_{dtype}_cycle{cycle_type}_vciter{maxiter_vcycle}_soliter{maxiter_solv}_k{vmap_k}_n{grid_size}_{method}"
    config = dict(solver=solver_name, coarse_solver=coarse_solver, dtype=dtype, tol=tol,
                    maxiter_vcycle=maxiter_vcycle, maxiter_solv=maxiter_solv, vmap_k=vmap_k,
                    cycle_type=cycle_type, grid_size=grid_size, method=method, smoother=smoother)
    save_results({"config": config, "time": time, "residual": residual}, f"{config_name}.json")


Config: ruge_stuben | jacobi | f64 | cycle=V | smoother=jacobi | n=50 | amjax_vmap
amjax_vmap-ruge_stuben/ time: 0.1246s/ residual: 3.18e-11
Results saved: /Users/fannymissillier/Desktop/AMJax/benchmarks/../results/ruge_stuben_jacobi_f64_cycleV_vciter100_soliter500_k64_n50_amjax_vmap.json

Config: ruge_stuben | jacobi | f64 | cycle=V | smoother=jacobi | n=50 | amjax_pcg_vmap
amjax_pcg_vmap-ruge_stuben/ time: 0.0755s/ residual: 6.16e-11
Results saved: /Users/fannymissillier/Desktop/AMJax/benchmarks/../results/ruge_stuben_jacobi_f64_cycleV_vciter100_soliter500_k64_n50_amjax_pcg_vmap.json

Config: ruge_stuben | jacobi | f64 | cycle=V | smoother=jacobi | n=100 | amjax_vmap
amjax_vmap-ruge_stuben/ time: 0.6158s/ residual: 4.62e-11
Results saved: /Users/fannymissillier/Desktop/AMJax/benchmarks/../results/ruge_stuben_jacobi_f64_cycleV_vciter100_soliter500_k64_n100_amjax_vmap.json

Config: ruge_stuben | jacobi | f64 | cycle=V | smoother=jacobi | n=100 | amjax_pcg_vmap
amjax_pcg_vmap-ruge_stub